# 01 clean national water points

Turns the WPdx extract into two national files that everything downstream reads.

Four exclusions define the analysis sample and three more define the candidate
set. Each was argued separately in the methods chapter, so the count at every
step is written out rather than only the endpoints.

Districts are taken from the district name the exchange records, harmonised
against the census spelling. No boundary file is read: nothing downstream needs
to know which polygon a point falls in, only how far it is from a demand cell,
and that is computed from coordinates in 07.

Districts whose extent changed after most records were collected are handled in
03, where the national comparison happens.

Writes `analysis_sample.csv` and `candidates_national.csv`. Nothing here is
specific to the study district; that comes in 04.

In [1]:
import sys
from pathlib import Path

# config.py sits beside the notebooks, so the working directory is enough. If a
# notebook is run from elsewhere, walk up until it is found.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))
from config import *


## 1. Cleaning cascade

In [2]:
import pandas as pd

USE = ['#clean_country_name', '#clean_adm2', '#clean_adm4',
       '#lat_deg', '#lon_deg',
       '#status_id', '#status_clean', '#management_clean',
       'is_urban', 'is_duplicate', 'usage_capacity',
       '#water_tech_category', '#install_year', '#report_date',
       'local_population_1km']

d = pd.read_csv(WPDX_CSV, usecols=USE, low_memory=False)
print(f"{len(d):,} records as downloaded")

98,774 records as downloaded


In [3]:
log = [("WPdx+ extract as downloaded", len(d))]

x = d[d['#clean_country_name'] == COUNTRY]
log.append((f"After restricting to {COUNTRY}", len(x)))

x = x[x.is_duplicate != True]
log.append(("After removing flagged duplicates", len(x)))

x = x[x.is_urban != True]
log.append(("After removing urban water points", len(x)))

x = x.dropna(subset=['#lat_deg', '#lon_deg'])
log.append(("After removing missing coordinates", len(x)))

x = x[x['#status_id'].isin(['Yes', 'No'])]
log.append(("Analysis sample", len(x)))

Two columns need repairing before they can be used.

In [4]:
# usage_capacity and local_population_1km carry thousands separators and read as
# text. Without stripping them every value above 999 silently becomes null.
for col in ('usage_capacity', 'local_population_1km'):
    x[col] = pd.to_numeric(
        x[col].astype(str).str.replace(',', '', regex=False), errors='coerce')

print(x[['usage_capacity', 'local_population_1km']].describe().round(0).to_string())

       usage_capacity  local_population_1km
count         79695.0               77118.0
mean            252.0                1735.0
std              97.0                1993.0
min              50.0                   0.0
25%             300.0                 654.0
50%             300.0                1195.0
75%             300.0                2043.0
max             300.0               27306.0


## 2. Candidate set

Three further exclusions, each argued in the methods chapter.

In [5]:
supply = x[x['#status_id'] == 'Yes']
cand = x[x['#status_id'] == 'No']
log.append(("  of which water available", len(supply)))
log.append(("  of which no water available", len(cand)))

cand_log = [("No water available", len(cand))]

cand = cand[cand['#status_clean'] != 'Abandoned/Decommissioned']
cand_log.append(("After removing abandoned or decommissioned", len(cand)))

cand = cand[cand['#status_clean'] != 'Non-Functional, dry season']
cand_log.append(("After removing dry season non-functional", len(cand)))

cand = cand[cand['#management_clean'] != 'Private Operator/Delegated Management']
cand_log.append(("After removing private or delegated management", len(cand)))

cand_log.append(("Candidate set", len(cand)))

## 3. District names

The exchange records a district for every point. Two are spelled differently
from the census tabulations; those are harmonised so that the two sources can
be joined in 03.

In [6]:
x["district"] = x["#clean_adm2"].str.upper().str.strip().replace(DISTRICT_SPELLING)
cand["district"] = cand["#clean_adm2"].str.upper().str.strip().replace(DISTRICT_SPELLING)

print(f"{x.district.nunique()} distinct districts in the analysis sample")
print(f"{int(x.district.isna().sum())} points with no district recorded")

134 distinct districts in the analysis sample
0 points with no district recorded


## 4. Write

In [7]:
x.to_csv(ANALYSIS_SAMPLE, index=False)
cand.to_csv(CANDIDATES_NAT, index=False)

full = log + [("", None)] + cand_log
pd.DataFrame(full, columns=["step", "records"]).to_csv(
    OUT / "01_cleaning_log.csv", index=False)

for k, v in full:
    print(f"{k:52s}{'' if v is None else v}")

WPdx+ extract as downloaded                         98774
After restricting to Uganda                         98767
After removing flagged duplicates                   97914
After removing urban water points                   95430
After removing missing coordinates                  95430
Analysis sample                                     92934
  of which water available                          76714
  of which no water available                       16220
                                                    
No water available                                  16220
After removing abandoned or decommissioned          15758
After removing dry season non-functional            15390
After removing private or delegated management      14635
Candidate set                                       14635


Expect 98,774 as downloaded, 92,934 in the analysis sample of which 76,714 have
water available, and 14,635 candidates nationally.